# Koschei Sentinel — Qwen3.5 9B Cyber SFT Smoke Run

This notebook performs a **real QLoRA weight update** on the versioned Defense Reflex v3 smoke curriculum. The resulting adapter is explicitly **smoke-only** and is not promotion-eligible until human-reviewed training data and the Cyber Range promotion gates are satisfied.

Before running: **Runtime → Change runtime type → GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pathlib, subprocess

repo = pathlib.Path('/content/drive/MyDrive/Koschei-Sentinel/runtime/koschei-sentinel')
repo.parent.mkdir(parents=True, exist_ok=True)
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/bugsbuny243/koschei-sentinel.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
print('Repo:', repo)
print('Commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import os, subprocess

env = os.environ.copy()
env['KOSCHEI_DRIVE_ROOT'] = '/content/drive/MyDrive/Koschei-Sentinel'
subprocess.run(
    ['bash', str(repo / 'scripts/run_cyber_sft_qwen35_9b_colab.sh'), str(repo)],
    check=True,
    env=env,
)


In [ ]:
import json, subprocess

run_rel = 'build/cyber-training/runs/qwen35-9b-smoke-001'
run_dir = repo / run_rel
verification = subprocess.check_output(
    ['sentinel-cyber-sft-verify', '--run-dir', run_rel],
    cwd=repo,
    text=True,
)
print(verification)
manifest = json.loads((run_dir / 'adapter-manifest.json').read_text())
receipt = json.loads((run_dir / 'training-receipt.json').read_text())
assert manifest['corpus_promotion_eligible'] is False
assert receipt['global_step'] > 0
print('Adapter SHA256:', manifest['adapter_digest'])
print('Receipt SHA256:', receipt['receipt_sha256'])
print('Global step:', receipt['global_step'])
print('Train metrics:', receipt['train_metrics'])
print('Eval metrics:', receipt['eval_metrics'])
print('Peak CUDA allocated GiB:', receipt['max_cuda_memory_allocated_gb'])
print('\nVERIFIED REAL SMOKE ADAPTER PRODUCED — promotion eligibility remains FALSE by design.')
